# 03 · Cell type annotation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/en/03_cell_type_annotation.ipynb)

**From Cells to Clusters: An Introduction to Single-Cell RNA-seq Analysis in Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Peru · November 9, 2026  
Instructor: Dr. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Time in the tutorial:** 11:00–11:35 (35 min)

**By the end of this notebook you will be able to:**
- Find marker genes for each cluster and annotate them manually with known markers
- Use a marker database (**PanglaoDB**) to score cell types
- Annotate automatically with a pre-trained classifier (**CellTypist**) and with a reference (`sc.tl.ingest`)
- Know where to find references and record annotations: **CZ CELLxGENE**, **Single Cell Portal**, **Cell Annotation Platform (CAP)**, Human Cell Atlas, Cell Ontology
- Compare annotation methods and decide on final labels

> 🟢 **Core** sections are covered live. 🔎 **Extra** sections are for self-study after the tutorial.

## 0. Setup 🟢
Run the cell below first (▶ or `Shift + Enter`). On Google Colab it installs the packages (~1 min);
on your own computer it assumes you already ran `pip install -r requirements.txt`.

**Optional:** set `USE_DRIVE = True` to save your results in Google Drive, so the next notebooks
can reuse them. If you leave it `False`, each notebook rebuilds what it needs automatically.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10" "decoupler>=2.1,<3" "celltypist>=1.7"

USE_DRIVE = False  # True = keep results in Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Tutorial helper functions (download + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # running from a local clone
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import decoupler as dc
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

In [ ]:
adata = tut.get_checkpoint("clustered", WORKDIR)
sc.pl.umap(adata, color="leiden", legend_loc="on data", title="Leiden clusters")

## 1. Three ways to annotate 🟢

| Strategy | How | Pros | Cons |
|---|---|---|---|
| **Manual (markers)** | Find genes enriched in each cluster, compare with known markers | Transparent, uses your expertise | Slow, subjective, needs good markers |
| **Marker databases** | Score each cell with curated marker lists (PanglaoDB, CellMarker) | Fast, no reference data needed | Lists are generic, tissue context matters |
| **Reference-based / classifiers** | Transfer labels from an annotated atlas (CellTypist, ingest, scANVI, Azimuth, SingleR) | Fast, reproducible, fine-grained | Only as good as the reference; cannot find new types |

In practice you **combine** them and use the agreement between methods as evidence.

## 2. Data-driven marker genes 🟢
`sc.tl.rank_genes_groups` compares each cluster against all other cells (Wilcoxon rank-sum test on the
log-normalized values stored in `adata.raw`).

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale="var")

In [ ]:
markers_df = sc.get.rank_genes_groups_df(adata, group=None)
top = (markers_df.query("logfoldchanges > 1 and pvals_adj < 0.05")
       .groupby("group", observed=True).head(5)
       .groupby("group", observed=True)["names"].apply(", ".join))
top.to_frame("top 5 markers")

## 3. Manual annotation with known markers 🟢
Classic PBMC markers (from the Seurat/Scanpy tutorials and the literature):

In [ ]:
pbmc_markers = tut.PBMC_MARKERS
pd.Series({k: ", ".join(v) for k, v in pbmc_markers.items()}).to_frame("markers")

In [ ]:
sc.pl.dotplot(adata, pbmc_markers, groupby="leiden", standard_scale="var")

**Reading a dot plot:** dot **size** = % of cells in the cluster expressing the gene; **color** = mean expression
(scaled per gene). A good marker is big and dark in one group and small elsewhere.

To make this reproducible, we **score** each marker set per cell (`sc.tl.score_genes`) and give each cluster the
label with the highest average score.

In [ ]:
adata = tut.annotate_by_markers(adata, pbmc_markers, cluster_key="leiden", key_added="cell_type")
score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
display(adata.obs.groupby("leiden", observed=True)[score_cols].mean().round(2))
pd.crosstab(adata.obs["leiden"], adata.obs["cell_type"])

If you disagree with a label (that is your expertise!), override it with a dictionary:
```python
manual = {"0": "CD4 T", "1": "CD14 Mono", ...}      # cluster -> cell type
adata.obs["cell_type"] = adata.obs["leiden"].map(manual).astype("category")
```

In [ ]:
sc.pl.umap(adata, color=["leiden", "cell_type"], legend_loc="on data", ncols=2)

## 4. Marker database: PanglaoDB 🟢
[**PanglaoDB**](https://panglaodb.se) curates >8,000 marker–cell type associations from >1,000 single-cell
datasets (human and mouse). We download it through **OmniPath** with `decoupler` and score every cell with a
**univariate linear model (ULM)**: for each cell type, how strongly do its markers explain the cell's expression?

In [ ]:
panglao = tut.cached_table("panglaodb_human", lambda: dc.op.resource("PanglaoDB", organism="human"), WORKDIR)
is_true = lambda s: s.astype(str).str.lower().isin(["true", "1", "1.0"])
markers_db = panglao[is_true(panglao["human"]) & is_true(panglao["canonical_marker"])
                     & (pd.to_numeric(panglao["human_sensitivity"], errors="coerce") > 0.5)]
markers_db = (markers_db.rename(columns={"cell_type": "source", "genesymbol": "target"})
              [["source", "target"]].drop_duplicates())
print(f"{markers_db['source'].nunique()} cell types, {len(markers_db)} markers")
markers_db.head()

In [ ]:
dc.mt.ulm(adata, net=markers_db, tmin=3, raw=True)
panglao_scores = dc.pp.get_obsm(adata, "score_ulm")
ranked = dc.tl.rankby_group(panglao_scores, groupby="leiden")
top_panglao = (ranked[ranked["stat"] > 0].sort_values("stat", ascending=False)
               .groupby("group", observed=True).head(3)
               .groupby("group", observed=True)["name"].apply(" | ".join))
top_panglao.to_frame("top 3 PanglaoDB cell types")

In [ ]:
best = ranked.sort_values("stat", ascending=False).groupby("group", observed=True).head(1).set_index("group")["name"]
adata.obs["panglaodb"] = adata.obs["leiden"].map(best).astype("category")
show = [c for c in ["T cells", "NK cells", "B cells", "Monocytes", "Dendritic cells", "Platelets"]
        if c in panglao_scores.var_names]
sc.pl.umap(panglao_scores, color=show, cmap="RdBu_r", vcenter=0, ncols=3)

## 5. Automated classifier: CellTypist 🟢
[**CellTypist**](https://www.celltypist.org) is a logistic-regression classifier trained on large immune atlases
(Domínguez Conde *et al.*, *Science* 2022). `Immune_All_Low` predicts ~98 fine-grained immune cell types;
`Immune_All_High` gives ~32 broader ones. Many other models exist (lung, gut, brain, developing tissues, mouse...).

CellTypist expects **log1p(counts per 10k)**, which is exactly what we stored in `adata.raw`.
`majority_voting=True` smooths the per-cell predictions within each cluster.

In [ ]:
import celltypist
from celltypist import models

model_name = "Immune_All_Low.pkl"
if not os.path.exists(models.get_model_path(model_name)):
    models.download_models(model=[model_name])

adata_ct = adata.raw.to_adata()     # all genes, log1p(CP10k)
pred = celltypist.annotate(adata_ct, model=model_name, majority_voting=True,
                           over_clustering="leiden")
adata.obs["celltypist"] = pred.predicted_labels["majority_voting"].astype("category").values
adata.obs["celltypist_conf"] = pred.probability_matrix.max(axis=1).values
sc.pl.umap(adata, color=["celltypist", "celltypist_conf"], ncols=2, wspace=0.6)

## 6. Reference mapping with `sc.tl.ingest` 🟢
`ingest` projects your cells onto the PCA/UMAP of an **annotated reference** and transfers the labels of the nearest
reference cells. Here the reference is `pbmc68k_reduced`, a small annotated subset of 68k PBMCs that ships with Scanpy.
The same idea scales to large atlases with **scANVI**, **scArches** or **Azimuth** (R).

In [ ]:
ref = sc.datasets.pbmc68k_reduced()
shared = ref.var_names.intersection(adata.var_names)
ref = ref[:, shared].copy()
query = adata[:, shared].copy()
n_pcs = min(30, len(shared) - 1)
sc.pp.pca(ref, n_comps=n_pcs)
sc.pp.neighbors(ref, n_pcs=n_pcs)
sc.tl.umap(ref)
sc.tl.ingest(query, ref, obs="bulk_labels")
adata.obs["ingest"] = query.obs["bulk_labels"].values
print(len(shared), "shared genes")
sc.pl.umap(adata, color="ingest")

## 7. Where to find references and record annotations 🟢

| Resource | What it offers | How to use it from Python |
|---|---|---|
| [**PanglaoDB**](https://panglaodb.se) | Curated marker genes (human, mouse) | `decoupler.op.resource("PanglaoDB")` (above) |
| [**CellMarker 2.0**](http://bio-bigdata.hrbmu.edu.cn/CellMarker/) | Manually curated markers from >100k publications | Download table → same ULM approach |
| [**CZ CELLxGENE Discover**](https://cellxgene.cziscience.com) | >1,000 standardized datasets, >100 million cells, all with Cell Ontology labels; *Explorer* for interactive browsing; *Gene Expression* tool to check a marker across tissues | `cellxgene_census` package to download annotated references as AnnData |
| [**Single Cell Portal**](https://singlecell.broadinstitute.org) (Broad Institute) | Hundreds of studies with interactive plots and downloadable matrices/metadata | Download `.h5ad`/matrices → reference for ingest/CellTypist training |
| [**Cell Annotation Platform (CAP)**](https://celltype.info) | Upload your `.h5ad`, record **each** cell type with marker evidence, synonyms and Cell Ontology terms; share/publish annotations | Export `adata.write_h5ad(...)` with labels + marker genes |
| [**Human Cell Atlas Data Portal**](https://data.humancellatlas.org) | Raw and processed data from HCA projects | Download data/references |
| [**CellTypist models**](https://www.celltypist.org/models) | Pre-trained classifiers for many tissues | `celltypist.models.download_models()` |
| [**Cell Ontology (CL)**](https://www.ebi.ac.uk/ols4/ontologies/cl) | Standard IDs for cell types (`CL:0000236` = B cell) | Store IDs in `adata.obs` so others can reuse your labels |

**Good practice:** store a Cell Ontology ID next to each label. CELLxGENE and CAP use these IDs, and they make
your annotations comparable across studies.

In [ ]:
cell_ontology = {
    "CD4 T": ("CD4-positive, alpha-beta T cell", "CL:0000624"),
    "CD8 T": ("CD8-positive, alpha-beta T cell", "CL:0000625"),
    "NK": ("natural killer cell", "CL:0000623"),
    "B": ("B cell", "CL:0000236"),
    "CD14 Mono": ("CD14-positive monocyte", "CL:0001054"),
    "FCGR3A Mono": ("CD14-low, CD16-positive monocyte", "CL:0002396"),
    "DC": ("dendritic cell", "CL:0000451"),
    "Platelet": ("platelet", "CL:0000233"),
}
adata.obs["cell_type_ontology_term"] = adata.obs["cell_type"].map({k: v[0] for k, v in cell_ontology.items()})
adata.obs["cell_type_ontology_term_id"] = adata.obs["cell_type"].map({k: v[1] for k, v in cell_ontology.items()})
adata.obs[["cell_type", "cell_type_ontology_term", "cell_type_ontology_term_id"]].drop_duplicates().reset_index(drop=True)

🔎 **Extra: downloading a reference from CZ CELLxGENE** (not run in class):
1. Browse <https://cellxgene.cziscience.com/datasets>, filter by tissue/disease, open a dataset and copy its **dataset ID**.
2. Download it as AnnData:
```python
!pip install -q cellxgene-census
import cellxgene_census
cellxgene_census.download_source_h5ad("<dataset_id>", to_path="reference.h5ad")
ref = sc.read_h5ad("reference.h5ad")      # labels in ref.obs["cell_type"]
```
Then use it with `sc.tl.ingest`, or train your own CellTypist model: `celltypist.train(ref, labels="cell_type")`.

## 8. Do the methods agree? 🟢
Agreement between independent methods is your evidence. Disagreement points to clusters that need a closer look
(mixed clusters, doublets, rare or activated states).

In [ ]:
sc.pl.umap(adata, color=["cell_type", "panglaodb", "celltypist", "ingest"], ncols=2, wspace=0.7)

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, other in zip(axes, ["panglaodb", "celltypist", "ingest"]):
    tab = pd.crosstab(adata.obs["cell_type"], adata.obs[other], normalize="index")
    sns.heatmap(tab, cmap="Blues", ax=ax, cbar=False, vmin=0, vmax=1)
    ax.set(title=f"markers vs {other}", xlabel=other, ylabel="cell_type (markers)")
plt.tight_layout()

## 9. Save the annotated object 🟢

In [ ]:
adata.uns.pop("rank_genes_groups", None)   # recomputed by cell type in notebook 04
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_annotated.h5ad")
adata.obs["cell_type"].value_counts()

### 💡 Your turn
Run CellTypist again with the coarser model `Immune_All_High.pkl` and compare it with `cell_type`
using `pd.crosstab`. Which labels become broader?

In [ ]:
# Your code here

<details><summary><b>Show solution</b></summary>

```python
models.download_models(model=["Immune_All_High.pkl"])
pred_high = celltypist.annotate(adata_ct, model="Immune_All_High.pkl", majority_voting=True,
                                over_clustering="leiden")
pd.crosstab(adata.obs["cell_type"], pred_high.predicted_labels["majority_voting"].values)
```
</details>

---
## ✅ Done!
Every cell has a name. Next: **04 · Visualization, differential expression and downstream analysis**.